# Day 3：从零手写 LLaMA 模型（本周最重要！）

> **目标**：从零实现完整的 LLaMA 模型，逐模块手写 `RMSNorm → RoPE → SwiGLU → GQA → LLaMABlock → LLaMA`，验证每个组件的正确性。

---

## 实现路线图

```
Part 1: 模型配置
  LLaMAConfig — 管理所有超参数

Part 2: RMSNorm
  去掉均值中心化，只保留 RMS 缩放 → 验证与 PyTorch LayerNorm 的差异

Part 3: RoPE 旋转位置编码
  precompute_freqs_cis → apply_rotary_emb → 验证相对位置性质

Part 4: SwiGLU FFN
  三权重矩阵 + SiLU 门控 → 验证输出形状

Part 5: Grouped Query Attention (GQA)
  支持 MHA / GQA / MQA 三种模式 → 验证 KV 头扩展

Part 6: LLaMA Block
  Pre-RMSNorm + GQA + RoPE + SwiGLU + Residual

Part 7: 完整 LLaMA 模型
  Token Embedding + N × Block + Final RMSNorm + LM Head

Part 8: 验证与测试
  参数量验证 + 前向传播测试 + 简单生成
```

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Part 1：模型配置

LLaMA 的配置与 GPT 有几个关键差异：
- `n_kv_heads`：KV 头数（GQA），当等于 `n_heads` 时退化为 MHA
- `ffn_dim_multiplier`：FFN 中间维度的调整因子
- `norm_eps`：RMSNorm 的 epsilon
- 无 `bias` 和 `dropout` 配置（LLaMA 不使用）

In [ ]:
@dataclass
class LLaMAConfig:
    """LLaMA 模型配置。"""
    dim: int = 4096              # 隐藏维度 d
    n_layers: int = 32           # Transformer 层数
    n_heads: int = 32            # Query 注意力头数
    n_kv_heads: int = 32         # KV 注意力头数（GQA，等于 n_heads 即 MHA）
    vocab_size: int = 32000      # 词表大小
    max_seq_len: int = 2048      # 最大序列长度
    norm_eps: float = 1e-5       # RMSNorm epsilon

    @property
    def head_dim(self) -> int:
        return self.dim // self.n_heads

    @property
    def ffn_dim(self) -> int:
        """SwiGLU FFN 中间维度：2/3 * 4d，取 256 的倍数。"""
        hidden = int(2 * self.dim * 4 / 3)
        return ((hidden + 255) // 256) * 256


# 用缩小版配置进行教学实验
config = LLaMAConfig(
    dim=512,
    n_layers=8,
    n_heads=8,
    n_kv_heads=4,       # GQA: 每 2 个 Q 头共享 1 组 KV
    vocab_size=32000,
    max_seq_len=256,
)

print(f"Config: {config}")
print(f"head_dim = {config.head_dim}")
print(f"ffn_dim = {config.ffn_dim}")
print(f"n_rep (Q heads per KV group) = {config.n_heads // config.n_kv_heads}")

## Part 2：RMSNorm

RMSNorm 相比 LayerNorm 去掉了均值中心化，只保留缩放：

$$\text{RMSNorm}(x) = \gamma \odot \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}}$$

- 没有 $\beta$（偏移参数）
- 没有减均值操作
- 可学习参数只有 $\gamma \in \mathbb{R}^d$

In [ ]:
class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))  # gamma

    def _norm(self, x: torch.Tensor) -> torch.Tensor:
        # x: (..., dim)
        # rsqrt = 1 / sqrt(x)
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output = self._norm(x.float()).type_as(x)
        return output * self.weight

In [ ]:
# 验证 RMSNorm
norm = RMSNorm(config.dim)
x = torch.randn(2, 10, config.dim)
y = norm(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {y.shape}")
print(f"Parameters:   {sum(p.numel() for p in norm.parameters())}")

# 手动验证计算
x_sample = x[0, 0]  # (dim,)
rms = torch.sqrt(x_sample.pow(2).mean() + norm.eps)
manual_result = (x_sample / rms) * norm.weight
auto_result = y[0, 0]
print(f"\nManual vs Forward max diff: {(manual_result - auto_result).abs().max().item():.2e}")

# 对比 LayerNorm
ln = nn.LayerNorm(config.dim)
print(f"\nLayerNorm params: {sum(p.numel() for p in ln.parameters())}")
print(f"RMSNorm params:   {sum(p.numel() for p in norm.parameters())}")
print(f"RMSNorm 参数量是 LayerNorm 的 {sum(p.numel() for p in norm.parameters()) / sum(p.numel() for p in ln.parameters()) * 100:.0f}%")

## Part 3：RoPE 旋转位置编码

RoPE 的核心：将 Q, K 向量按位置旋转，使得它们的内积只依赖相对位置。

步骤：
1. 预计算频率 `freqs_cis = e^{i * m * theta}`（复数形式）
2. 将 Q, K 的实数向量视为复数向量
3. 乘以 `freqs_cis` 完成旋转

频率参数：$\theta_k = 10000^{-2k/d}, \quad k = 0, 1, \ldots, d/2-1$

In [ ]:
def precompute_freqs_cis(dim: int, seq_len: int, theta: float = 10000.0) -> torch.Tensor:
    """
    预计算 RoPE 的复数频率。

    Args:
        dim: 每个注意力头的维度 (head_dim)
        seq_len: 最大序列长度
        theta: RoPE 的基础频率

    Returns:
        freqs_cis: (seq_len, dim//2) complex tensor
    """
    # theta_k = 10000^{-2k/d}, k = 0, 1, ..., d/2-1
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    # 位置索引 m = 0, 1, ..., seq_len-1
    t = torch.arange(seq_len, dtype=torch.float32)
    # 外积: (seq_len,) × (dim/2,) → (seq_len, dim/2)
    freqs = torch.outer(t, freqs)
    # 转为复数: e^{i * m * theta_k}
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # magnitude=1, angle=freqs
    return freqs_cis


def reshape_for_broadcast(freqs_cis: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    """调整 freqs_cis 的形状以便广播到 (batch, seq_len, n_heads, head_dim/2)。"""
    ndim = x.ndim
    assert ndim >= 2
    shape = [1] * ndim
    shape[1] = x.shape[1]   # seq_len
    shape[-1] = x.shape[-1] # head_dim/2
    return freqs_cis.view(*shape)


def apply_rotary_emb(
    xq: torch.Tensor,
    xk: torch.Tensor,
    freqs_cis: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    对 Q 和 K 应用旋转位置编码。

    Args:
        xq: (batch, seq_len, n_heads, head_dim)
        xk: (batch, seq_len, n_kv_heads, head_dim)
        freqs_cis: (seq_len, head_dim//2) complex

    Returns:
        xq_out, xk_out: 旋转后的 Q, K
    """
    # 将实数向量 reshape 为复数: (B, T, H, D) → (B, T, H, D/2) complex
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))

    # 调整 freqs_cis 形状并旋转
    freqs_cis_q = reshape_for_broadcast(freqs_cis, xq_)
    freqs_cis_k = reshape_for_broadcast(freqs_cis, xk_)

    xq_out = torch.view_as_real(xq_ * freqs_cis_q).flatten(-2)
    xk_out = torch.view_as_real(xk_ * freqs_cis_k).flatten(-2)

    return xq_out.type_as(xq), xk_out.type_as(xk)

In [ ]:
# 验证 RoPE
freqs_cis = precompute_freqs_cis(config.head_dim, config.max_seq_len)
print(f"freqs_cis shape: {freqs_cis.shape}")
print(f"freqs_cis dtype: {freqs_cis.dtype}")
print(f"freqs_cis[0,:4]: {freqs_cis[0,:4]}  (位置 0, 全是 1+0j 即不旋转)")
print(f"freqs_cis[1,:4]: {freqs_cis[1,:4]}  (位置 1, 开始旋转)")

# 验证旋转不改变向量长度
xq = torch.randn(1, 10, config.n_heads, config.head_dim)
xk = torch.randn(1, 10, config.n_kv_heads, config.head_dim)
xq_rot, xk_rot = apply_rotary_emb(xq, xk, freqs_cis[:10])

print(f"\nQ norm before RoPE: {xq[0,0,0].norm().item():.4f}")
print(f"Q norm after  RoPE: {xq_rot[0,0,0].norm().item():.4f}")
print(f"RoPE preserves vector norm: {torch.allclose(xq.norm(dim=-1), xq_rot.norm(dim=-1), atol=1e-5)}")

# 验证相对位置性质
# q at pos m, k at pos n → dot product should only depend on (m-n)
q_single = torch.randn(1, 1, 1, config.head_dim).expand(1, 20, 1, -1)
k_single = torch.randn(1, 1, 1, config.head_dim).expand(1, 20, 1, -1)
q_rot, k_rot = apply_rotary_emb(q_single, k_single, freqs_cis[:20])

# dot(q[5], k[3]) should equal dot(q[7], k[5]) (both have relative distance 2)
dot_5_3 = (q_rot[0, 5, 0] * k_rot[0, 3, 0]).sum().item()
dot_7_5 = (q_rot[0, 7, 0] * k_rot[0, 5, 0]).sum().item()
print(f"\ndot(q[5], k[3]) = {dot_5_3:.6f}")
print(f"dot(q[7], k[5]) = {dot_7_5:.6f}")
print(f"Same relative distance, same dot product: {abs(dot_5_3 - dot_7_5) < 1e-4}")

## Part 4：SwiGLU FFN

$$\text{SwiGLU}(x) = (\text{SiLU}(x W_{\text{gate}}) \otimes x W_{\text{up}}) \cdot W_{\text{down}}$$

三个权重矩阵，无 bias：
- $W_{\text{gate}} \in \mathbb{R}^{d \times d_{ff}}$ — 门控投影
- $W_{\text{up}} \in \mathbb{R}^{d \times d_{ff}}$ — 上投影
- $W_{\text{down}} \in \mathbb{R}^{d_{ff} \times d}$ — 下投影

In [ ]:
class FeedForward(nn.Module):
    """SwiGLU Feed-Forward Network."""

    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.w_gate = nn.Linear(dim, hidden_dim, bias=False)
        self.w_up   = nn.Linear(dim, hidden_dim, bias=False)
        self.w_down = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # SwiGLU: SiLU(x @ W_gate) * (x @ W_up) → W_down
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

In [ ]:
# 验证 SwiGLU FFN
ffn = FeedForward(config.dim, config.ffn_dim)
x = torch.randn(2, 10, config.dim)
y = ffn(x)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {y.shape}")
print(f"FFN dim:      {config.ffn_dim}")
print(f"Expected:     {int(2 * config.dim * 4 / 3 + 255) // 256 * 256}")

# 参数量
ffn_params = sum(p.numel() for p in ffn.parameters())
print(f"\nSwiGLU FFN params: {ffn_params:,}")
print(f"  w_gate: {config.dim} × {config.ffn_dim} = {config.dim * config.ffn_dim:,}")
print(f"  w_up:   {config.dim} × {config.ffn_dim} = {config.dim * config.ffn_dim:,}")
print(f"  w_down: {config.ffn_dim} × {config.dim} = {config.ffn_dim * config.dim:,}")

# 对比标准 FFN
standard_ffn_params = 2 * config.dim * (4 * config.dim)
print(f"\n标准 GELU FFN (4d) params: {standard_ffn_params:,}")
print(f"SwiGLU FFN params:          {ffn_params:,}")
print(f"比值: {ffn_params / standard_ffn_params:.3f}")

## Part 5：Grouped Query Attention (GQA)

GQA 允许多个 Q 头共享同一组 KV，减少 KV Cache 开销。

关键实现细节：
1. Q 投影到 `n_heads × head_dim`，K/V 投影到 `n_kv_heads × head_dim`
2. 在计算 Attention 前，将 KV 头重复 `n_rep = n_heads // n_kv_heads` 次
3. 对 Q 和 K 应用 RoPE
4. 使用因果掩码

In [ ]:
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    将 KV 头重复 n_rep 次以匹配 Q 头数。

    Args:
        x: (batch, seq_len, n_kv_heads, head_dim)
        n_rep: 每组的 Q 头数

    Returns:
        (batch, seq_len, n_kv_heads * n_rep, head_dim)
    """
    if n_rep == 1:
        return x
    B, T, n_kv_heads, head_dim = x.shape
    return (
        x[:, :, :, None, :]
        .expand(B, T, n_kv_heads, n_rep, head_dim)
        .reshape(B, T, n_kv_heads * n_rep, head_dim)
    )


class Attention(nn.Module):
    """Grouped Query Attention with RoPE."""

    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.n_heads = config.n_heads
        self.n_kv_heads = config.n_kv_heads
        self.n_rep = self.n_heads // self.n_kv_heads
        self.head_dim = config.head_dim

        self.wq = nn.Linear(config.dim, config.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(config.dim, config.n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(config.dim, config.n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(config.n_heads * self.head_dim, config.dim, bias=False)

    def forward(
        self,
        x: torch.Tensor,
        freqs_cis: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        B, T, _ = x.shape

        # 线性投影
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim)

        # 应用 RoPE（只对 Q 和 K，不对 V）
        q, k = apply_rotary_emb(q, k, freqs_cis)

        # GQA: 扩展 KV 头以匹配 Q 头
        k = repeat_kv(k, self.n_rep)  # (B, T, n_heads, head_dim)
        v = repeat_kv(v, self.n_rep)

        # 转置为 (B, n_heads, T, head_dim) 以便矩阵乘法
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Scaled Dot-Product Attention
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores + mask

        attn_weights = F.softmax(scores.float(), dim=-1).type_as(q)
        output = torch.matmul(attn_weights, v)

        # 合并多头: (B, n_heads, T, head_dim) → (B, T, dim)
        output = output.transpose(1, 2).contiguous().reshape(B, T, -1)
        return self.wo(output)

In [ ]:
# 验证 GQA
attn = Attention(config)
x = torch.randn(2, 10, config.dim)
freqs_cis_slice = freqs_cis[:10]

# 创建因果掩码
T = 10
mask = torch.full((T, T), float('-inf'))
mask = torch.triu(mask, diagonal=1)
mask = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, T, T)

y = attn(x, freqs_cis_slice, mask)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {y.shape}")

# 参数量分析
print(f"\n=== Attention 参数量 ===")
print(f"wq: {config.dim} × {config.n_heads * config.head_dim} = {config.dim * config.n_heads * config.head_dim:,}")
print(f"wk: {config.dim} × {config.n_kv_heads * config.head_dim} = {config.dim * config.n_kv_heads * config.head_dim:,}")
print(f"wv: {config.dim} × {config.n_kv_heads * config.head_dim} = {config.dim * config.n_kv_heads * config.head_dim:,}")
print(f"wo: {config.n_heads * config.head_dim} × {config.dim} = {config.n_heads * config.head_dim * config.dim:,}")
print(f"Total: {sum(p.numel() for p in attn.parameters()):,}")

# GQA 节省的参数
mha_kv_params = 2 * config.dim * config.n_heads * config.head_dim
gqa_kv_params = 2 * config.dim * config.n_kv_heads * config.head_dim
print(f"\nMHA KV params: {mha_kv_params:,}")
print(f"GQA KV params: {gqa_kv_params:,}")
print(f"KV 参数节省: {(1 - gqa_kv_params / mha_kv_params) * 100:.0f}%")

## Part 6：LLaMA Block

一个 LLaMA Block = Pre-RMSNorm + GQA(+RoPE) + Residual + Pre-RMSNorm + SwiGLU + Residual

$$h_l' = h_l + \text{GQA}(\text{RMSNorm}(h_l))$$
$$h_{l+1} = h_l' + \text{SwiGLU}(\text{RMSNorm}(h_l'))$$

In [ ]:
class TransformerBlock(nn.Module):
    """LLaMA Transformer Block."""

    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.attention_norm = RMSNorm(config.dim, eps=config.norm_eps)
        self.attention = Attention(config)
        self.ffn_norm = RMSNorm(config.dim, eps=config.norm_eps)
        self.feed_forward = FeedForward(config.dim, config.ffn_dim)

    def forward(
        self,
        x: torch.Tensor,
        freqs_cis: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        # Pre-RMSNorm + Attention + Residual
        x = x + self.attention(self.attention_norm(x), freqs_cis, mask)
        # Pre-RMSNorm + FFN + Residual
        x = x + self.feed_forward(self.ffn_norm(x))
        return x

In [ ]:
# 验证 LLaMA Block
block = TransformerBlock(config)
x = torch.randn(2, 10, config.dim)
y = block(x, freqs_cis[:10], mask)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {y.shape}")

block_params = sum(p.numel() for p in block.parameters())
print(f"Block params: {block_params:,}")
print(f"  Attention:  {sum(p.numel() for p in block.attention.parameters()):,}")
print(f"  FFN:        {sum(p.numel() for p in block.feed_forward.parameters()):,}")
print(f"  Norms:      {sum(p.numel() for p in block.attention_norm.parameters()) + sum(p.numel() for p in block.ffn_norm.parameters()):,}")

## Part 7：完整 LLaMA 模型

Token Embedding → N × TransformerBlock → Final RMSNorm → LM Head

与 GPT 的关键差异：
1. 没有 Position Embedding（RoPE 在 Attention 中处理）
2. LM Head 不与 Token Embedding 共享权重
3. 所有 Linear 层无 bias

In [ ]:
class LLaMA(nn.Module):
    """完整的 LLaMA 模型。"""

    def __init__(self, config: LLaMAConfig):
        super().__init__()
        self.config = config

        self.tok_embeddings = nn.Embedding(config.vocab_size, config.dim)
        self.layers = nn.ModuleList(
            [TransformerBlock(config) for _ in range(config.n_layers)]
        )
        self.norm = RMSNorm(config.dim, eps=config.norm_eps)
        self.output = nn.Linear(config.dim, config.vocab_size, bias=False)

        # 预计算 RoPE 频率
        self.freqs_cis = precompute_freqs_cis(
            config.head_dim, config.max_seq_len
        )

        # 初始化权重
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        tokens: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Args:
            tokens: (batch, seq_len) token IDs
            targets: (batch, seq_len) target token IDs (for training)

        Returns:
            logits: (batch, seq_len, vocab_size)
            loss: scalar (if targets provided)
        """
        B, T = tokens.shape
        assert T <= self.config.max_seq_len, f"Sequence length {T} exceeds max {self.config.max_seq_len}"

        # Token Embedding（无位置编码！）
        h = self.tok_embeddings(tokens)  # (B, T, dim)

        # 取对应长度的 RoPE 频率
        freqs_cis = self.freqs_cis[:T].to(h.device)

        # 因果掩码
        mask = torch.full((T, T), float('-inf'), device=h.device)
        mask = torch.triu(mask, diagonal=1)
        mask = mask.unsqueeze(0).unsqueeze(0)  # (1, 1, T, T)

        # N 层 Transformer Block
        for layer in self.layers:
            h = layer(h, freqs_cis, mask)

        # Final RMSNorm + LM Head
        h = self.norm(h)
        logits = self.output(h)  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
            )

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
    ) -> torch.Tensor:
        """自回归生成（无 KV Cache 的简单版本，Day 5/6 将加入 Cache）。"""
        for _ in range(max_new_tokens):
            # 截断到最大长度
            idx_cond = idx if idx.size(1) <= self.config.max_seq_len else idx[:, -self.config.max_seq_len:]

            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)

        return idx

In [ ]:
# 创建模型并验证
model = LLaMA(config)
print(f"=== LLaMA Model (教学版) ===")
print(f"Config: dim={config.dim}, layers={config.n_layers}, heads={config.n_heads}, kv_heads={config.n_kv_heads}")

# 参数量统计
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,} ({total_params / 1e6:.1f}M)")

# 逐组件参数量
emb_params = model.tok_embeddings.weight.numel()
layer_params = sum(p.numel() for p in model.layers.parameters())
norm_params = sum(p.numel() for p in model.norm.parameters())
head_params = model.output.weight.numel()

print(f"\n组件参数量分解:")
print(f"  Token Embedding: {emb_params:>12,} ({emb_params/total_params*100:.1f}%)")
print(f"  {config.n_layers} × Block:      {layer_params:>12,} ({layer_params/total_params*100:.1f}%)")
print(f"  Final RMSNorm:   {norm_params:>12,} ({norm_params/total_params*100:.1f}%)")
print(f"  LM Head:         {head_params:>12,} ({head_params/total_params*100:.1f}%)")

In [ ]:
# 前向传播测试
tokens = torch.randint(0, config.vocab_size, (2, 20))
targets = torch.randint(0, config.vocab_size, (2, 20))

logits, loss = model(tokens, targets)
print(f"Input shape:   {tokens.shape}")
print(f"Logits shape:  {logits.shape}")
print(f"Loss:          {loss.item():.4f}")
print(f"Expected initial loss ≈ ln({config.vocab_size}) = {math.log(config.vocab_size):.4f}")

# 简单生成测试
prompt = torch.randint(0, config.vocab_size, (1, 5))
generated = model.generate(prompt, max_new_tokens=10, temperature=1.0, top_k=50)
print(f"\nPrompt length:    {prompt.shape[1]}")
print(f"Generated length: {generated.shape[1]}")
print(f"Generated tokens: {generated[0].tolist()}")

## Part 8：与 LLaMA-7B 真实参数量对比

让我们用 LLaMA-7B 的真实配置来验证参数量计算。

In [ ]:
# LLaMA-7B 真实配置参数量计算
config_7b = LLaMAConfig(
    dim=4096,
    n_layers=32,
    n_heads=32,
    n_kv_heads=32,  # LLaMA-1 用 MHA
    vocab_size=32000,
    max_seq_len=2048,
)

d = config_7b.dim
N = config_7b.n_layers
V = config_7b.vocab_size
d_ff = config_7b.ffn_dim
n_h = config_7b.n_heads
n_kv = config_7b.n_kv_heads
d_k = config_7b.head_dim

print(f"=== LLaMA-7B 参数量计算 ===")
print(f"d={d}, N={N}, V={V}, d_ff={d_ff}, n_h={n_h}, n_kv={n_kv}, d_k={d_k}")

# 单层参数
attn_params = d * (n_h * d_k) + d * (n_kv * d_k) * 2 + (n_h * d_k) * d  # wq + wk + wv + wo
ffn_params = 3 * d * d_ff  # w_gate + w_up + w_down
norm_params_per_layer = 2 * d  # 2 × RMSNorm
layer_total = attn_params + ffn_params + norm_params_per_layer

print(f"\n--- 单层参数量 ---")
print(f"Attention (wq+wk+wv+wo): {attn_params:>15,}")
print(f"SwiGLU FFN (3 matrices): {ffn_params:>15,}")
print(f"RMSNorm × 2:             {norm_params_per_layer:>15,}")
print(f"Layer total:             {layer_total:>15,}")

# 完整模型
emb = V * d
all_layers = N * layer_total
final_norm = d
lm_head = d * V  # 不共享
total = emb + all_layers + final_norm + lm_head

print(f"\n--- 完整模型参数量 ---")
print(f"Token Embedding:  {emb:>15,}")
print(f"{N} × Layer:       {all_layers:>15,}")
print(f"Final RMSNorm:    {final_norm:>15,}")
print(f"LM Head:          {lm_head:>15,}")
print(f"Total:            {total:>15,} ({total/1e9:.2f}B)")
print(f"\n论文报告: 6.7B → 我们计算: {total/1e9:.2f}B ✓")

## Part 9：模型架构打印

查看完整的模型结构，确认每一层的参数配置。

In [ ]:
# 打印教学版模型结构
print(model)
print("\n" + "="*60)
print("模型结构对比 (GPT vs LLaMA):")
print("="*60)
print(f"{'组件':<25} {'GPT':<25} {'LLaMA':<25}")
print("-"*75)
print(f"{'Embedding':<25} {'tok_emb + pos_emb':<25} {'tok_emb only':<25}")
print(f"{'Normalization':<25} {'LayerNorm':<25} {'RMSNorm':<25}")
print(f"{'Position Encoding':<25} {'Learnable absolute':<25} {'RoPE (in Attention)':<25}")
print(f"{'Attention':<25} {'MHA':<25} {'GQA + RoPE':<25}")
print(f"{'FFN':<25} {'GELU, 2 matrices':<25} {'SwiGLU, 3 matrices':<25}")
print(f"{'Bias':<25} {'Yes':<25} {'No':<25}")
print(f"{'Weight Tying':<25} {'Yes (Emb=Head)':<25} {'No':<25}")

## 总结

本 notebook 实现了完整的 LLaMA 模型，包含所有相比 GPT 的架构改进：

| 组件 | 实现状态 | 关键点 |
|------|---------|-------|
| RMSNorm | ✅ | 去掉均值中心化，只有 gamma 参数 |
| RoPE | ✅ | 复数旋转，作用于 Q/K，编码相对位置 |
| SwiGLU FFN | ✅ | 三权重矩阵 + SiLU 门控，d_ff = 2/3 × 4d |
| GQA | ✅ | KV 头数可少于 Q 头数，减少 Cache |
| LLaMA Block | ✅ | Pre-RMSNorm + GQA + SwiGLU + Residual |
| LLaMA Model | ✅ | 无 pos_emb，无 bias，不共享权重 |

**接下来**：
- Day 4：深入 RoPE 的复数推导和外推性
- Day 5：加入 KV Cache 实现高效推理
- Day 6：在小规模数据上跑通 LLaMA 预训练